# Experiment №2

This notebook walks through the process of applying the Diffusion Lens technique
to analyze the intermediate steps of a diffusion model fine-tuned using Dreambooth.

**Goal:** We need to explore how in laten space the concepts of specified token `xon` correlates with the `dog`.


In [1]:
import sys
import os

parent_dir = os.path.abspath(os.path.join('..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
import torch
from transformers import CLIPTextModel, CLIPTokenizer

## Configuration

In [3]:
# --- Model Paths ---
MODEL_BASE = "runwayml/stable-diffusion-v1-5" # Base model used for Dreambooth training
TEXT_ENCODER_PATH = "../outputs/dreambooth_dog/text_encoder_final.pt" # Path to your fine-tuned text encoder weights file

In [5]:
# Load models for experiment
tokenizer = CLIPTokenizer.from_pretrained(MODEL_BASE, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_BASE, subfolder="text_encoder")
text_encoder_tuned = CLIPTextModel.from_pretrained(MODEL_BASE, subfolder="text_encoder")

try:
    # Load Text Encoder weights
    print(f"Loading fine-tuned Text Encoder weights from: {TEXT_ENCODER_PATH}")
    text_encoder_state_dict = torch.load(TEXT_ENCODER_PATH, map_location="cpu")
    if not any("module." in k for k in text_encoder_state_dict.keys()):
        text_encoder_tuned.load_state_dict(text_encoder_state_dict)
    else:
        print("  (Handling 'module.' prefix in Text Encoder state_dict)")
        new_state_dict_text_encoder = OrderedDict()
        for k, v in text_encoder_state_dict.items():
            name = k[7:] if k.startswith("module.") else k
            new_state_dict_text_encoder[name] = v
        text_encoder_tuned.load_state_dict(new_state_dict_text_encoder)
    print("Text Encoder weights loaded.")

except FileNotFoundError as e:
    print(f"Error: Could not find state dict file: {e}")
    raise
except Exception as e:
    print(f"Error loading state dicts: {e}")
    raise

Loading fine-tuned Text Encoder weights from: ../outputs/dreambooth_dog/text_encoder_final.pt
Text Encoder weights loaded.


# Explore latent space of text encoder

In [13]:
from torch.nn.functional import cosine_similarity
import torch

def compute_the_similarity(tokenizer, text_encoder, prompt_specific, prompt_general, specific_token_str, general_token_str):
    # --- Tokenize ---
    inputs_specific = tokenizer(prompt_specific, return_tensors="pt", padding=True)
    inputs_general = tokenizer(prompt_general, return_tensors="pt", padding=True)

    # Find 'xon' token ID 
    specific_token_id = tokenizer(specific_token_str, add_special_tokens=False).input_ids[0]
    specific_token_index = inputs_specific.input_ids[0].tolist().index(specific_token_id)

    # Find 'dog' token ID 
    dog_token_id_general = tokenizer(general_token_str, add_special_tokens=False).input_ids[0]
    dog_token_index_general = inputs_general.input_ids[0].tolist().index(dog_token_id_general)

    # --- Get Embeddings ---
    with torch.no_grad():
        outputs_specific = text_encoder(**inputs_specific)
        outputs_general = text_encoder(**inputs_general)

    # Last hidden state shape: (batch_size, sequence_length, hidden_size)
    last_hidden_state_specific = outputs_specific.last_hidden_state
    last_hidden_state_general = outputs_general.last_hidden_state

    # --- Extract Embeddings for the specific tokens ---
    embedding_xon = last_hidden_state_specific[0, specific_token_index, :]
    embedding_dog = last_hidden_state_general[0, dog_token_index_general, :]

    # Compare 'xon' from specific prompt with 'dog' from general prompt
    similarity_xon_vs_dog_general = cosine_similarity(embedding_xon.unsqueeze(0), embedding_dog.unsqueeze(0))
    print(f"Cosine Similarity ('{specific_token_str}' vs '{general_token_str}'): {similarity_xon_vs_dog_general.item():.4f}")

In [7]:
# Define the concepts for exploration
prompt_specific = "xon"
prompt_general = "dog"
specific_token_str = "xon" 
general_token_str = "dog"

In [14]:
compute_the_similarity(tokenizer,text_encoder_tuned,prompt_specific,prompt_general,specific_token_str,general_token_str)

Cosine Similarity ('xon' vs 'dog'): 0.2504


In [15]:
compute_the_similarity(tokenizer,text_encoder,prompt_specific,prompt_general,specific_token_str,general_token_str)

Cosine Similarity ('xon' vs 'dog'): 0.1781


# Conclusion

We observe that DreamBooth fine‑tuning succeeds in pulling our learned token `xon` closer to its intended concept `dog` in the model’s latent space. Quantitatively, the cosine similarity between the tuned embedding of “xon” and the generic “dog” prompt rises from 0.1781 (untuned) to 0.2504 (tuned). This roughly **40% increase in similarity** indicates that the fine‑tuning process has meaningfully reduced the distance between the general concept and our personalized token.

In practice, this means that, after tuning, the model is more likely to treat `xon` as a genuine instance of `dog` (and its associated attributes) rather than as an unrelated or ambiguous placeholder. In other words, the deeper embedding layers have absorbed the DreamBooth adjustments to reflect our specificity, yielding a token representation that aligns more closely with the semantic space of `dog`. Moving forward, one could inspect which layers contribute most to this shift, or experiment with alternative similarity measures, to further dissect how and where DreamBooth imprints custom concepts into the diffusion pipeline.